# Self-supervised pretraining monitor (TimesNet, benign + GridSybil, T = 64)

This notebook only **reads** a pretraining run folder (`config.json`, `env.json`, `metrics.jsonl`) written by
`python -m src.model.benign_gridsybil.timesnet.train` and plots it. It never trains, never loads a checkpoint and never
writes anything. Each section says in plain words what the plot shows and what a healthy run looks like.

Choose the run in the next cell: leave `RUN_NAME = None` for the newest run folder that has a `metrics.jsonl`, or set it
to a folder name under `src/runs/pretraining/benign_gridsybil/T64/` (the only folder this notebook lists). A run that is still
going (no summary yet) or that lacks some keys is fine: missing pieces are reported and skipped.

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RUN_NAME: str | None = None  # e.g. "smoke-timesnet"; None = newest run with metrics.jsonl
RUNS_SUBDIRS = (Path("src/runs/pretraining/benign_gridsybil/T64"),)  # the only runs folder listed
LENGTH_ONLY_AUC = 0.531  # length-only bar from the encoder-input EDA


def find_repo_root(start: Path) -> Path:
    """Walks up from `start` to the folder holding CLAUDE.md and agent.md (the repo root)."""
    for folder in [start, *start.parents]:
        if (folder / "CLAUDE.md").exists() and (folder / "agent.md").exists():
            return folder
    raise FileNotFoundError("repo root (CLAUDE.md + agent.md) not found above " + str(start))


ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(ROOT)  # repo-root contract (RULE 5); every path below is relative to it
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("repo root:", ROOT)

## Loading a run

`RunRecord` holds one run folder: the settings, the environment, and the metric lines split by `kind`
(`train` every `log_every` steps, `eval` at init / each epoch / stop, `summary` once at the end).

In [ ]:
class RunRecord:
    """Read-only view of one pretraining run folder.

    Attributes:
        path: The run folder.
        config: Contents of config.json ({} if absent).
        env: Contents of env.json ({} if absent).
        train: One row per logged training step.
        evals: One row per evaluation (scalar keys only; list-valued keys stay in `eval_records`).
        eval_records: The raw evaluation dictionaries.
        summary: The summary line, or None while the run has not finished.
    """

    def __init__(self, path: Path) -> None:
        self.path = path
        self.config = self._read_json(path / "config.json")
        self.env = self._read_json(path / "env.json")
        records = self._read_metrics(path / "metrics.jsonl")
        self.train = pd.DataFrame([r for r in records if r.get("kind") == "train"])
        self.eval_records = [r for r in records if r.get("kind") == "eval"]
        self.evals = pd.DataFrame(
            [{k: v for k, v in r.items() if not isinstance(v, (list, dict))} for r in self.eval_records]
        )
        summaries = [r for r in records if r.get("kind") == "summary"]
        self.summary = summaries[-1] if summaries else None

    @staticmethod
    def _read_json(path: Path) -> dict[str, Any]:
        return json.loads(path.read_text()) if path.exists() else {}

    @staticmethod
    def _read_metrics(path: Path) -> list[dict[str, Any]]:
        records = []
        for line in path.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:  # a line being written right now
                print("skipped an unreadable line:", line[:80])
        return records

    @staticmethod
    def available(runs_dirs: list[Path]) -> list[Path]:
        """Run folders with a metrics.jsonl across `runs_dirs`, oldest first (earlier dirs win on a name clash)."""
        seen: dict[str, Path] = {}
        for runs_dir in runs_dirs:
            for metrics in runs_dir.glob("*/metrics.jsonl"):
                seen.setdefault(metrics.parent.name, metrics.parent)
        return sorted(seen.values(), key=lambda p: (p / "metrics.jsonl").stat().st_mtime)

    @classmethod
    def pick(cls, runs_dirs: list[Path], name: str | None) -> "RunRecord":
        """Opens the folder called `name` (first of `runs_dirs` that has it), or the newest run with a metrics.jsonl."""
        if name is not None:
            for runs_dir in runs_dirs:
                if (runs_dir / name).is_dir():
                    return cls(runs_dir / name)
            raise FileNotFoundError(f"no run folder {name!r} under {[str(d) for d in runs_dirs]}")
        candidates = cls.available(runs_dirs)
        if not candidates:
            raise FileNotFoundError(f"no run with metrics.jsonl under {[str(d) for d in runs_dirs]}")
        return cls(candidates[-1])

    def train_col(self, key: str) -> pd.Series | None:
        """A training column indexed by step, or None if never logged."""
        if self.train.empty or key not in self.train:
            return None
        return self.train.set_index("step")[key].dropna()

    def eval_col(self, key: str) -> pd.Series | None:
        """An evaluation column indexed by step, or None if never logged."""
        if self.evals.empty or key not in self.evals:
            return None
        return self.evals.set_index("step")[key].dropna()

    def feature_names(self) -> list[str]:
        """The 13 feature names from the encoder-input metadata (generic names if it cannot be read)."""
        meta = Path(self.config.get("data_dir", "")) / "metadata.json"
        try:
            return list(json.loads(meta.read_text())["features"])
        except (OSError, KeyError, json.JSONDecodeError):
            return [f"f{i}" for i in range(int(self.config.get("n_features", 13)))]


def plot_series(ax: plt.Axes, run: RunRecord, keys: list[str], source: str = "train", **kw: Any) -> list[str]:
    """Draws the available keys on `ax`; returns the keys that were missing."""
    missing = []
    for key in keys:
        series = run.train_col(key) if source == "train" else run.eval_col(key)
        if series is None or series.empty:
            missing.append(key)
            continue
        ax.plot(series.index, series.values, marker="o" if source == "eval" else None, ms=4, label=key, **kw)
    if len(keys) - len(missing):
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "not logged", ha="center", va="center", transform=ax.transAxes, color="grey")
    return missing


runs_dirs = [ROOT / d for d in RUNS_SUBDIRS]
run = RunRecord.pick(runs_dirs, RUN_NAME)
print("run:", run.path.name, "in", run.path.parent)
print(f"train lines {len(run.train)}, eval lines {len(run.evals)}, summary {'yes' if run.summary else 'not yet'}")
print("available runs:", [f"{p.parent.parent.name}/{p.name}" for p in RunRecord.available(runs_dirs)])

## Run settings and environment

What the run was asked to do (`config.json`) and where it ran (`env.json`: versions, device, data hashes, and the
**pre-flight** checks done before the first step). In the pre-flight, every `padding_invariance_*` value should be
**0**: adding padding rows must not change any loss or the embedding z.

In [ ]:
groups = {
    "data": ["data_dir", "shards_per_split", "seq_len", "n_features"],
    "encoder": ["d_model", "d_ff", "n_blocks", "top_k", "n_kernels", "dropout", "proj_dim"],
    "objectives": [
        "use_recon",
        "use_nce",
        "use_physics",
        "lambda_recon",
        "lambda_nce",
        "lambda_p1",
        "lambda_p2",
        "lambda_p3",
        "recon_ratio",
        "temperature",
        "inject_prob",
        "norm_warmup_steps",
    ],
    "optimisation": [
        "batch_size",
        "lr",
        "weight_decay",
        "warmup_steps",
        "grad_clip",
        "max_epochs",
        "max_steps",
        "max_hours",
        "patience",
        "seed",
    ],
    "resources": ["device", "mem_gb", "workers", "eval_windows"],
}
rows = [(g, k, run.config[k]) for g, keys in groups.items() for k in keys if k in run.config]
shown = {k for _, k, _ in rows}
rows += [("other", k, v) for k, v in run.config.items() if k not in shown]
display(pd.DataFrame(rows, columns=["group", "setting", "value"]).set_index(["group", "setting"]))

env_rows = {k: v for k, v in run.env.items() if not isinstance(v, dict)}
display(pd.Series(env_rows, name="environment", dtype=object).to_frame())
for key in ("rows", "shards"):
    if key in run.env:
        print(f"{key}:", run.env[key])
preflight = run.env.get("preflight", {})
if preflight:
    display(pd.Series(preflight, name="preflight").to_frame())
    bad = {k: v for k, v in preflight.items() if k.startswith("padding_invariance") and abs(v) > 1e-5}
    print("padding invariance:", "OK (all 0)" if not bad else f"NOT ZERO {bad}")
else:
    print("no preflight block in env.json")

## Training losses

Raw (unscaled) losses logged during training, plus the learning rate, the gradient norm before clipping, seconds per
step and memory. In a healthy run the losses trend down (noisy from batch to batch), the learning rate warms up then
follows a cosine, the gradient norm settles below the clip value, and memory stays flat below the budget.
`same_link_pairs` counts batch pairs from the same link (possible false negatives for the contrastive loss).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
panels = [
    ("joint loss (weighted, scaled)", ["joint"]),
    ("reconstruction (masked MSE)", ["recon"]),
    ("contrastive (NT-Xent)", ["nce"]),
    ("physics heads P1-P3 (BCE)", ["p1", "p2", "p3"]),
    ("learning rate", ["lr"]),
    ("gradient norm (before clipping)", ["grad_norm"]),
]
missing = []
for ax, (title, keys) in zip(axes.ravel(), panels):
    missing += plot_series(ax, run, keys)
    ax.set_title(title)
    ax.set_xlabel("step")
if "grad_clip" in run.config and run.train_col("grad_norm") is not None:
    axes[1, 2].axhline(run.config["grad_clip"], ls="--", c="k", lw=1, label="clip")
    axes[1, 2].legend(fontsize=8)
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, (title, key) in zip(
    axes,
    [("seconds per step", "step_seconds"), ("memory (GB)", "mem_gb"), ("same-link pairs in batch", "same_link_pairs")],
):
    missing += plot_series(ax, run, [key])
    ax.set_title(title)
    ax.set_xlabel("step")
if "mem_gb" in run.config and run.train_col("mem_gb") is not None:
    axes[1].axhline(run.config["mem_gb"], ls="--", c="r", lw=1, label="budget")
    axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()
print("missing training keys:", missing or "none")

## Pretrain-check losses vs the interpolation baseline

The same losses measured on the held-out pretrain-check windows (vehicles never used for training; no labels read).
The dashed line is a **free baseline**: fill each masked row by linear interpolation from its neighbours. The model's
reconstruction should drop **below** that line; if it stays above, the encoder is doing worse than a ruler.
The `init` point is the random-initialised model, a reference for "nothing learned yet".

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
miss = plot_series(axes[0], run, ["val_recon"], source="eval")
base = run.eval_col("val_recon_interp_baseline")
if base is not None and not base.empty:
    axes[0].axhline(base.iloc[-1], ls="--", c="k", lw=1, label="interpolation baseline")
    axes[0].legend(fontsize=8)
axes[0].set_title("pretrain-check reconstruction")
miss += plot_series(axes[1], run, ["val_nce", "val_joint"], source="eval")
axes[1].set_title("pretrain-check contrastive / joint")
miss += plot_series(axes[2], run, ["val_p1", "val_p2", "val_p3"], source="eval")
axes[2].axhline(math.log(2), ls=":", c="grey", lw=1)
axes[2].set_title("pretrain-check P1-P3 BCE (dotted = chance, ln 2)")
for ax in axes:
    ax.set_xlabel("step")
fig.tight_layout()
plt.show()

cols = [
    c
    for c in [
        "tag",
        "step",
        "epoch",
        "val_joint",
        "val_recon",
        "val_recon_interp_baseline",
        "val_nce",
        "val_p1",
        "val_p2",
        "val_p3",
        "scales_frozen",
        "eval_seconds",
    ]
    if c in run.evals
]
if cols:
    display(run.evals[cols].round(4))
print("missing eval keys:", miss or "none")

## Collapse monitors

Self-supervised encoders can "collapse": every window maps to (almost) the same z, which makes the contrastive loss
look fine while the embedding is useless. Signs of health:

- **effective rank** of z stays well above 1 (it counts how many directions z really uses; max = d),
- **z std** (per dimension, mean and minimum) stays clearly above 0,
- **alignment** (distance between two augmented views, lower = better) is small while **uniformity** (lower = spread
  out) goes down,
- **cos_pos** (similarity of two views of the same window) stays high while **cos_neg** (different windows) falls
  toward 0. If both are high together, the embedding has collapsed.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
miss = plot_series(axes[0], run, ["z_effective_rank"], source="eval")
axes[0].set_title(f"effective rank of z (d = {run.config.get('d_model', '?')})")
miss += plot_series(axes[1], run, ["z_std_mean", "z_std_min"], source="eval")
axes[1].set_title("z standard deviation")
miss += plot_series(axes[2], run, ["align", "uniformity"], source="eval")
axes[2].set_title("alignment / uniformity")
miss += plot_series(axes[3], run, ["cos_pos", "cos_neg"], source="eval")
axes[3].set_title("cosine: same window vs different")
for ax in axes:
    ax.set_xlabel("step")
fig.tight_layout()
plt.show()
print("missing collapse keys:", miss or "none")

## Physics heads: can they spot the violations we inject?

P1 (position jump), P2 (speed change) and P3 (heading flip) are trained to detect violations **we inject** into
otherwise real windows (decision D4); real attack labels are never used. AUROC on the pretrain-check windows measures
how well each head separates injected from clean windows: **0.5 = guessing**, 1.0 = perfect. They should rise above 0.5
after training starts.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
miss = plot_series(ax, run, ["auroc_p1", "auroc_p2", "auroc_p3"], source="eval")
ax.axhline(0.5, ls="--", c="k", lw=1)
ax.set_ylim(0.3, 1.02)
ax.set_title("P-head AUROC on injected violations (dashed = chance)")
ax.set_xlabel("step")
plt.show()
print("missing P-head keys:", miss or "none")

## Length shortcut monitors

Windows have different numbers of real messages (the rest is padding). If z mostly encodes **how long** a window is,
later probes could separate classes through length alone (the encoder-input EDA found a length-only AUC of
**0.531**, i.e. length alone barely separates benign from GridSybil). Two monitors:

- `length_probe_r2`: how well a ridge regression on z predicts the window length. Some length information is
  natural; a value near 1 means length is plainly readable from z, so later downstream probes must be compared against
  the length-only AUC bar of 0.531 (a probe that does not beat it learned nothing beyond length).
- `corr_znorm_length`: correlation between the size of z and the length; near 0 is good (masked pooling should
  stop length from leaking into the scale of z).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
miss = plot_series(axes[0], run, ["length_probe_r2"], source="eval")
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_title("length predictable from z (ridge R²)")
miss += plot_series(axes[1], run, ["corr_znorm_length"], source="eval")
axes[1].axhline(0, ls="--", c="k", lw=1)
axes[1].set_ylim(-1, 1)
axes[1].set_title("corr(|z|, length)")
for ax in axes:
    ax.set_xlabel("step")
fig.tight_layout()
plt.show()
print(f"bar for later probes: length-only AUC = {LENGTH_ONLY_AUC}")
print("missing length keys:", miss or "none")

## Reconstruction per feature vs interpolation

The latest pretrain-check reconstruction error for each of the 13 features (normalised units), next to the
interpolation baseline and the random-init model. Smooth features (positions) are easy for interpolation, so the model
may stay above the baseline there for a while; the interesting features are the ones interpolation cannot guess
(acceleration, log-Δτ, bearing).

In [ ]:
with_feat = [r for r in run.eval_records if isinstance(r.get("val_recon_per_feature"), list)]
if not with_feat:
    print("no per-feature reconstruction logged")
else:
    names = run.feature_names()
    last, first = with_feat[-1], with_feat[0]
    table = pd.DataFrame({"feature": names, f"model ({last.get('tag', 'last')})": last["val_recon_per_feature"]})
    if first is not last:
        table[f"model ({first.get('tag', 'first')})"] = first["val_recon_per_feature"]
    if isinstance(last.get("val_recon_interp_baseline_per_feature"), list):
        table["interpolation"] = last["val_recon_interp_baseline_per_feature"]
    table = table.set_index("feature")
    ax = table.plot.bar(figsize=(13, 4), width=0.8)
    ax.set_ylabel("masked MSE (normalised)")
    ax.set_title("reconstruction error per feature")
    plt.tight_layout()
    plt.show()
    if "interpolation" in table:
        model_col = table.columns[0]
        table["model beats interpolation"] = table[model_col] < table["interpolation"]
    display(table.round(4))

## Run summary and stop reason

The final line of `metrics.jsonl`. Stop reasons: `max_epochs` (all epochs done), `early_stopping` (pretrain-check
joint loss stopped improving for `patience` epochs), `max_steps` (step cap, used by the smoke run), `time_budget`,
`memory_budget` (the 16 GB guard). No summary means the run is still going or was interrupted; `last.pt` then holds
the latest epoch for `--resume`.

In [ ]:
if run.summary is None:
    print("no summary yet: the run is still going or was interrupted")
    if not run.train.empty:
        print(f"last logged step {int(run.train['step'].max())}, epoch {int(run.train['epoch'].max())}")
else:
    display(pd.Series(run.summary, name="summary").to_frame())
if not run.evals.empty and "val_joint" in run.evals:
    best = run.evals.loc[run.evals["val_joint"].idxmin()]
    print(f"best pretrain-check joint {best['val_joint']:.4f} at step {int(best['step'])} ({best.get('tag', '')})")
print("checkpoints present:", sorted(p.name for p in run.path.glob("*.pt")) or "none")

## How to read a healthy run (checklist)

The cell below checks the latest evaluation against simple rules. They are rules of thumb for spotting a broken run,
not pass marks for the thesis.

| Check | Healthy |
|---|---|
| pre-flight padding invariance | all 0 |
| pretrain-check reconstruction | falls below the interpolation baseline over training |
| training losses | trend down, no NaN, gradient norm mostly below the clip |
| effective rank of z | well above 1 and not falling toward 1 |
| z std (min) | clearly above 0 |
| cos_pos vs cos_neg | cos_pos high, cos_neg near 0 (gap large) |
| P-head AUROC | above 0.5 and rising |
| corr(\|z\|, length) | near 0 |
| memory | flat, below the budget |

In [ ]:
def health_checks(run: RunRecord) -> pd.DataFrame:
    """Rule-of-thumb checks on the latest evaluation; 'n/a' when a key was not logged."""
    last = run.eval_records[-1] if run.eval_records else {}
    pre = run.env.get("preflight", {})

    def rule(name: str, value: Any, ok: bool | None, healthy: str) -> dict[str, Any]:
        status = "n/a" if ok is None else ("ok" if ok else "look")
        return {"check": name, "value": value, "healthy": healthy, "status": status}

    def get(key: str) -> float | None:
        v = last.get(key)
        return float(v) if isinstance(v, (int, float)) and not isinstance(v, bool) else None

    pad = [v for k, v in pre.items() if k.startswith("padding_invariance")]
    rows = [rule("padding invariance (max)", max(pad) if pad else None, (max(pad) <= 1e-5) if pad else None, "0")]
    recon, base = get("val_recon"), get("val_recon_interp_baseline")
    rows.append(
        rule(
            "recon / interpolation",
            None if recon is None or not base else round(recon / base, 3),
            None if recon is None or not base else recon < base,
            "< 1",
        )
    )
    rank = get("z_effective_rank")
    rows.append(rule("effective rank of z", rank, None if rank is None else rank > 5, "> 5"))
    zmin = get("z_std_min")
    rows.append(rule("z std (min)", zmin, None if zmin is None else zmin > 0.05, "> 0.05"))
    cp, cn = get("cos_pos"), get("cos_neg")
    rows.append(
        rule(
            "cos_pos - cos_neg",
            None if cp is None or cn is None else round(cp - cn, 3),
            None if cp is None or cn is None else cp - cn > 0.3,
            "> 0.3",
        )
    )
    for k in ("auroc_p1", "auroc_p2", "auroc_p3"):
        v = get(k)
        rows.append(rule(k, v, None if v is None else v > 0.55, "> 0.55"))
    corr = get("corr_znorm_length")
    rows.append(rule("|corr(|z|, length)|", corr, None if corr is None else abs(corr) < 0.3, "< 0.3 in size"))
    finite = None
    if not run.train.empty:
        num = run.train.select_dtypes("number")
        finite = bool(np.isfinite(num.to_numpy()).all())
    rows.append(rule("training values finite", finite, finite, "True"))
    mem = run.train_col("mem_gb")
    budget = run.config.get("mem_gb")
    rows.append(
        rule(
            "peak memory (GB)",
            None if mem is None else float(mem.max()),
            None if mem is None or budget is None else float(mem.max()) < budget,
            f"< {budget}",
        )
    )
    return pd.DataFrame(rows).set_index("check")


print("latest evaluation:", run.eval_records[-1].get("tag") if run.eval_records else "none")
display(health_checks(run))